# Project 19 — BROKEN notebook (debugging exercise)

Seeded bugs centred on the ODE pitfall: **practical non-identifiability** of $k$ and $V$ when the design is poor. Run it, read the $(k,V)$ pair plot and diagnostics, find each bug, fix it. Answer key: `BROKEN_BUGS.md`.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import pytensor.tensor as pt
RNG = 20240601

In [ ]:
from data.generate_data import generate
full = generate()
# BUG 1: keep ONLY late timepoints (>= 3h). Without early points, C0=D/V is
#         unconstrained, so V and k become practically non-identifiable.
mask = full['t'] >= 3.0
data = {'t': full['t'][mask], 'y': full['y'][mask], 'dose': full['dose']}
print('using', mask.sum(), 'of', full['n'], 'timepoints (late only)')

### Model — vague priors on top of a bad design make it worse.

In [ ]:
# BUG 2: vague priors on log k and log V (sigma=3) remove the last bit of
#         information that could have rescued the poor design.
t = data['t']; y = data['y']; dose = data['dose']
with pm.Model() as model:
    log_k = pm.Normal('log_k', mu=0.0, sigma=3.0)
    log_V = pm.Normal('log_V', mu=0.0, sigma=3.0)
    k = pm.Deterministic('k', pt.exp(log_k))
    V = pm.Deterministic('V', pt.exp(log_V))
    sigma = pm.HalfNormal('sigma', sigma=1.0)
    C = (dose/V)*pt.exp(-k*t)
    pm.Normal('y_obs', mu=C, sigma=sigma, observed=y)
    idata = pm.sample(draws=500, tune=800, chains=2, cores=1,
                      target_accept=0.9, random_seed=RNG, progressbar=False)

In [ ]:
print(az.summary(idata, var_names=['k','V','sigma']))
print('divergences:', int(idata.sample_stats['diverging'].sum()))

### The smoking gun — BUG 3: the (k, V) pair plot is a ridge.

Without early timepoints, many $(k, V)$ pairs fit the late-time data equally well: a long diagonal correlation, inflated R-hat, low ESS. Note that the *product* (clearance $k V$) may still be okay while $k$ and $V$ separately are not — the hallmark of practical non-identifiability. Fixes: (1) restore the early timepoints (design); (2) use the informative priors from `model.py`.

In [ ]:
az.plot_pair(idata, var_names=['k','V'], kind='scatter',
             scatter_kwargs={'alpha':0.2}); plt.tight_layout()
kk = idata.posterior['k'].values.ravel(); VV = idata.posterior['V'].values.ravel()
print('corr(k,V) =', round(float(np.corrcoef(kk,VV)[0,1]),3))